# PPAP Quality Review Agent — Google Colab Demo

**Do not paste this `.ipynb` file into a code cell.** Open it as a notebook:
- [Open in Google Colab](https://colab.research.google.com/github/rockyforever8-sys/Agentic-MDS/blob/cursor/ppap-langgraph-prototype-17d5/PPAP_Colab_Start_Here.ipynb)
- Or Colab **File → Upload notebook**

Animated **LangGraph** workflow for automotive SQE PPAP review automation.

| Scenario | PPAP ID | Expected |
|----------|---------|----------|
| Clean accept | `PPAP-2026-001` | Accept |
| Missing docs | `PPAP-2026-002` | Hold |
| Critical dim OOS | `PPAP-2026-003` | Reject |
| PFMEA RPN issue | `PPAP-2026-004` | Hold |
| Cpk failure | `PPAP-2026-006` | Reject |

In [ ]:
# Cell 1 — Clone repo and install dependencies
import os, pathlib, subprocess, sys

ROOT = pathlib.Path('/content/Agentic-MDS')
REPO = 'https://github.com/rockyforever8-sys/Agentic-MDS.git'
REF = os.environ.get('PPAP_GIT_REF', 'cursor/ppap-langgraph-prototype-17d5')

if not (ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REF, REPO, str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', REF])
    subprocess.check_call(['git', '-C', str(ROOT), 'checkout', '-B', REF, f'origin/{REF}'])

sys.path.insert(0, str(ROOT))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'langgraph', 'langchain-core', 'rich'])
print(f'Ready: {ROOT}')

In [ ]:
# Cell 2 — Seed synthetic PPAP database
import os
from ppap_agent.database.seed import seed_database
from ppap_agent.database.db import list_pending_ppaps

DB = ROOT / 'ppap_agent' / 'data' / 'ppap_synthetic.db'
summary = seed_database(DB)
os.environ['PPAP_DB_PATH'] = str(DB)

print(f'Seeded: {summary["ppap_submissions"]} PPAPs, {summary["aiag_rules"]} AIAG rules')
print(f'\nInbox ({len(list_pending_ppaps())} pending):')
for p in list_pending_ppaps():
    print(f'  {p["id"]}  {p["part_number"]:20s}  {p["supplier_name"]}')

## Animated Single PPAP Review

Watch the LangGraph nodes execute step-by-step. Change `PPAP_ID` to try different scenarios:
- `PPAP-2026-001` → clean accept
- `PPAP-2026-003` → critical dimension reject
- `PPAP-2026-002` → missing documents hold

In [ ]:
# Cell 3 — Animated graph visualization
import time
from IPython.display import HTML, clear_output, display
from ppap_agent.visualization import render_graph_html, stream_ppap_review

PPAP_ID = 'PPAP-2026-003'  # ← change to try other scenarios
DELAY = 0.7  # seconds between steps

final_state = {}
for step in stream_ppap_review(PPAP_ID):
    clear_output(wait=True)
    display(HTML(render_graph_html(
        active_nodes=step['active_nodes'],
        completed_nodes=step['completed_nodes'],
        ppap_id=PPAP_ID,
        decision=step.get('state', {}).get('decision') if step.get('done') else None,
        risk_band=step.get('state', {}).get('risk_band') if step.get('done') else None,
    )))
    print(f"▸ {step['message']}")
    final_state = step.get('state', final_state)
    if not step.get('done'):
        time.sleep(DELAY)

d = final_state.get('decision', '?')
print(f'\n{"="*50}')
print(f'DECISION: {d.upper()}  |  Risk: {final_state.get("risk_band")} ({final_state.get("risk_score", 0):.0f}/100)')
for r in final_state.get('decision_reasons', []):
    print(f'  • {r}')
print(f'\nSupplier: {final_state.get("supplier_notification", "")}')

## Batch Supervisor Graph

Re-seed the database (previous review marked submissions as processed), then run the supervisor graph across all inbox items.

In [ ]:
# Cell 4 — Batch process all pending PPAPs
seed_database(DB)  # reset to pending
from ppap_agent.agents.batch_graph import run_batch_review

result = run_batch_review(max_reviews=8)
s = result['batch_summary']

print(f'Batch complete: {s["reviews_completed"]} reviews')
print(f'  Accepted: {s["accepted"]}  |  Rejected: {s["rejected"]}  |  On Hold: {s["on_hold"]}')
print(f'  Auto-accept rate: {s.get("auto_accept_rate", 0)}%')
print()
for c in result['completed']:
    icon = {'accept': '✅', 'reject': '❌', 'hold': '⏸️'}.get(c['decision'], '?')
    print(f'  {icon} {c["ppap_id"]}  {c["part_number"]:20s}  {c["decision"].upper():6s}  risk={c["risk_score"]:.0f}')

## One-Button Full Demo

Runs the complete demo script: animated review + batch processing.

In [ ]:
# Cell 5 — One-button full demo
%run colab_ppap_demo.py

## Compare All Scenarios

Run every synthetic scenario and show decision matrix.

In [ ]:
# Cell 6 — Decision matrix for all 8 scenarios
from ppap_agent.agents.graph import run_ppap_review

scenarios = [
    'PPAP-2026-001', 'PPAP-2026-002', 'PPAP-2026-003', 'PPAP-2026-004',
    'PPAP-2026-005', 'PPAP-2026-006', 'PPAP-2026-007', 'PPAP-2026-008',
]

print(f'{"PPAP ID":<16} {"Part":<20} {"Decision":<8} {"Risk":<6} {"Score":>5} {"Findings":>8}')
print('-' * 70)
for pid in scenarios:
    seed_database(DB)
    r = run_ppap_review(pid)
    icon = {'accept': '✅', 'reject': '❌', 'hold': '⏸️'}.get(r['decision'], '?')
    print(f'{pid:<16} {r.get("part_number",""):<20} {icon} {r["decision"]:<6} {r.get("risk_band",""):<6} {r.get("risk_score",0):5.0f} {len(r.get("all_findings",[])):>8}')